In [ ]:
import os
import sys
import json
import logging
from datetime import datetime
from pathlib import Path
import yaml
import pandas as pd

# Set base pipeline directory
TASK_DIR = Path.cwd().resolve()
if TASK_DIR.name != "DAY15 TASK":
    TASK_DIR = TASK_DIR / "DAY15 TASK"
if str(TASK_DIR) not in sys.path:
    sys.path.insert(0, str(TASK_DIR))

print(f"Pipeline Root Directory: {TASK_DIR}")

In [ ]:
CONFIG_PATH = TASK_DIR / "config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("Configured Datasets:")
for file_name, mappings in config.get("column_mapping", {}).items():
    print(f"  • {file_name}: {len(mappings)} column mappings")

In [ ]:
log_dir = TASK_DIR / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

log_filename = f"pipeline_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.log"
log_filepath = log_dir / log_filename

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_filepath, encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
    force=True,
)

logger = logging.getLogger("OlistPipeline")
logger.info("Olist pipeline logging initialized.")

In [ ]:
output_dir = TASK_DIR / "output"
output_dir.mkdir(parents=True, exist_ok=True)
manifest_path = output_dir / "manifest.json"


def load_manifest(path: Path) -> dict:
    if path.exists() and path.stat().st_size > 0:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"processed_files": []}


def record_processed_file(path: Path, filename: str):
    manifest = load_manifest(path)
    if filename not in manifest["processed_files"]:
        manifest["processed_files"].append(filename)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=4)
        logger.info(f"Manifest saved successfully: {filename}")


manifest = load_manifest(manifest_path)
print(f"Currently processed files in manifest: {manifest['processed_files']}")

In [ ]:
class SchemaError(Exception):
    pass


def check_schema(df: pd.DataFrame, required_columns: list[str]) -> bool:
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise SchemaError(f"Missing required columns: {missing}")
    return True


def validate_price(price: float) -> bool:
    try:
        return float(price) > 0
    except (ValueError, TypeError):
        return False

In [ ]:
def process_dataset(file_path: Path, mapping: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = pd.read_csv(file_path)
    df = df.rename(columns=mapping)

    bad_mask = pd.Series(False, index=df.index)
    if "item_price" in df.columns:
        bad_mask |= ~df["item_price"].apply(validate_price)
    if "shipping_cost" in df.columns:
        bad_mask |= (df["shipping_cost"] == 0.0)

    bad_records = df[bad_mask].copy()
    valid_records = df[~bad_mask].copy()
    bad_records["source_file"] = file_path.name

    return valid_records, bad_records


def run_olist_pipeline():
    logger.info("Olist pipeline started")
    data_dir = TASK_DIR / "data"
    manifest = load_manifest(manifest_path)
    column_mappings = config.get("column_mapping", {})

    all_bad_records = []

    for file_name, mapping in column_mappings.items():
        file_path = data_dir / file_name
        if not file_path.exists():
            continue

        logger.info(f"Processing file: {file_name}")
        try:
            valid_df, bad_df = process_dataset(file_path, mapping)
            if not bad_df.empty:
                all_bad_records.append(bad_df)
                logger.warning(f"{file_name}: Found {len(bad_df)} anomalous records.")

            record_processed_file(manifest_path, file_name)
        except Exception as exc:
            logger.error(f"Failed to process {file_name}: {exc}")

    if all_bad_records:
        combined_bad = pd.concat(all_bad_records, ignore_errors=True)
        bad_records_path = output_dir / "bad_records.csv"
        combined_bad.to_csv(bad_records_path, index=False)
        logger.info(f"Saved {len(combined_bad)} bad records to {bad_records_path.name}")

    logger.info("Olist pipeline completed")


run_olist_pipeline()

In [ ]:
!pytest tests/ -v